# 🧠 Exploring Mental Health Data 

# 🎯 Objective

This notebook aims to analyze a dataset on mental health with the goal of building a binary classification model to predict the likelihood of depression. Depression, a prevalent mental health issue, can be influenced by factors such as academic and work pressures, sleep duration, family history, and personal habits. This study seeks to understand the relationships between these factors and depression, supporting better mental health interventions.

![image.png](https://arc-anglerfish-washpost-prod-washpost.s3.amazonaws.com/public/LMCWWU2OTBFSBE4EOHMBMSVFI4.PNG)
Source: Neon Genesis Evangelion

# 1. 📋 Introduction

Depression is a mental health condition that affects a significant portion of the global population, influencing daily life, work, and academic performance. Early detection and intervention are crucial for improving mental health outcomes. In this project, we will leverage a dataset containing various features related to demographics, lifestyle, and work/study pressures to predict the presence of depression.

Problem Statement:
We aim to predict the target variable, Depression, which is a binary variable indicating whether an individual shows signs of depression (1) or not (0). This will be approached as a binary classification problem using machine learning models.


In [ ]:
import pandas
import numpy
import os

import catboost
import lightgbm
import plotly.express
import plotly.graph_objects
import sklearn.preprocessing
import sklearn.ensemble
import xgboost

DATA_DIR = "/kaggle/input/playground-series-s4e11"

# 2. 📊 Data Overview


In [ ]:
TRAINING_DATAFRAME = pandas.read_csv(os.path.join(DATA_DIR, "train.csv"))
TESTING_DATAFRAME  = pandas.read_csv(os.path.join(DATA_DIR, "test.csv"))

TRAINING_DATAFRAME.columns

The dataset includes the following features:

🧍 Demographic Information: `Name`, `Gender`, `Age`, `City`

💼 Academic/Work Status: `Working Professional or Student`, `Profession`, `Degree`

🌱 Lifestyle and Satisfaction: `Sleep Duration`, `Dietary Habits`, `Study Satisfaction`, `Job Satisfaction`

⚖️ Pressure and Stress Factors: `Academic Pressure`, `Work Pressure`, `Financial Stress`

🏥 Health Indicators: `Family History of Mental Illness`, `Have you ever had suicidal thoughts?`

Our target variable is `Depression`, which we will predict based on these features.

In [ ]:
TRAINING_DATAFRAME.head()

In [ ]:
TRAINING_DATAFRAME.describe()

In [ ]:
TRAINING_DATAFRAME.describe(include='object')

# 3. 🚀 Project Workflow

## 3.1 🧹 Data Preprocessing: Clean and preprocess the data, handle missing values, encode categorical features, and scale numerical features as necessary.

In [ ]:
TRAINING_DATAFRAME.info()

In [ ]:
TRAINING_DATAFRAME.isnull().sum()

Missing values in `Profession` can be filled with the help of some other data in the dataset. Because `Working Professional or Student` is complete, it can be put in the `Profession` column where the data is missing.

In [ ]:
TRAINING_DATAFRAME.loc[TRAINING_DATAFRAME["Profession"].isnull(), "Profession"] = TRAINING_DATAFRAME["Working Professional or Student"].loc[TRAINING_DATAFRAME["Profession"].isnull()]

Missing values of `Academic Pressure` can be replaced by 0 for "Working Professionals" and the median value for "Students". In a similar way, missing values for `Work Pressure` can be replaced by 0 for "Students" and the median value for "Working Professionals".

I know this assumption fails in the case where a student might be working as a part time student, and so I have not considered that case.

In [ ]:
TRAINING_DATAFRAME.loc[(TRAINING_DATAFRAME["Academic Pressure"].isnull()) & (TRAINING_DATAFRAME["Working Professional or Student"] == "Student"), 
                       "Academic Pressure"] = TRAINING_DATAFRAME["Academic Pressure"].median()
TRAINING_DATAFRAME.loc[(TRAINING_DATAFRAME["Academic Pressure"].isnull()) & (TRAINING_DATAFRAME["Working Professional or Student"] == "Working Professional"), 
                       "Academic Pressure"] = 0
TRAINING_DATAFRAME.loc[(TRAINING_DATAFRAME["Work Pressure"].isnull()) & (TRAINING_DATAFRAME["Working Professional or Student"] == "Working Professional"),
                       "Work Pressure"] = TRAINING_DATAFRAME["Work Pressure"].median()
TRAINING_DATAFRAME.loc[(TRAINING_DATAFRAME["Work Pressure"].isnull()) & (TRAINING_DATAFRAME["Working Professional or Student"] == "Student"),
                       "Work Pressure"] = 0

In order to fill `CGPA`, `Study Satisfaction`, `Job Satisfaction` and `Financial Stress`, the mean value is used.

In [ ]:
for col in ["CGPA", "Study Satisfaction", "Job Satisfaction", "Financial Stress"]:
    TRAINING_DATAFRAME[col] = TRAINING_DATAFRAME[col].fillna(TRAINING_DATAFRAME[col].mean())

Missing Values in `Dietary Habits` and `Degree` are filled with "UNKNOWN"

In [ ]:
for col in ["Dietary Habits", "Degree"]:
    TRAINING_DATAFRAME[col] = TRAINING_DATAFRAME[col].fillna("UNKNOWN")

In [ ]:
def __preprocess_missing_values(dataframe):
    dataframe.loc[dataframe["Profession"].isnull(), "Profession"] = dataframe["Working Professional or Student"].loc[dataframe["Profession"].isnull()]
    dataframe.loc[(dataframe["Academic Pressure"].isnull()) & (dataframe["Working Professional or Student"] == "Student"), 
                       "Academic Pressure"] = dataframe["Academic Pressure"].median()
    dataframe.loc[(dataframe["Academic Pressure"].isnull()) & (dataframe["Working Professional or Student"] == "Working Professional"), 
                           "Academic Pressure"] = 0
    dataframe.loc[(dataframe["Work Pressure"].isnull()) & (dataframe["Working Professional or Student"] == "Working Professional"),
                           "Work Pressure"] = dataframe["Work Pressure"].median()
    dataframe.loc[(dataframe["Work Pressure"].isnull()) & (dataframe["Working Professional or Student"] == "Student"),
                           "Work Pressure"] = 0
    
    for col in ["CGPA", "Study Satisfaction", "Job Satisfaction", "Financial Stress"]:
        dataframe[col] = dataframe[col].fillna(dataframe[col].mean())
    
    for col in ["Dietary Habits", "Degree"]:
        dataframe[col] = dataframe[col].fillna("UNKNOWN")
    
    return dataframe

TESTING_DATAFRAME = __preprocess_missing_values(TESTING_DATAFRAME)
TESTING_DATAFRAME.isnull().sum()

Now that the missing values are replaced, its time to clean up the data.

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["City"].value_counts().reset_index(), 
    x = "City", 
    y = "count", 
    labels={'index': 'Category', 'Category': 'Frequency'}, 
    title = "Frequency of City",
    template="plotly_dark").show()

There are a number of people who have incorrectly filled in the city. In order to fix this, a city's name is cross referenced with OSM (Open Street Maps).

In [ ]:
# import requests

# def is_a_city(city_name):
#     headers = {
#         'User-Agent': 'something/1.0 (something@something.com)'
#     }
    
#     url = "https://nominatim.openstreetmap.org/search"
#     params = {
#         'city': city_name,
#         'country': 'India',
#         'format': 'json'
#     }
    
#     response = requests.get(url, params=params, headers=headers)
    
#     if response.status_code == 200:
#         data = response.json()
#         print(data)
        
#         for _ in data:
#             if 'addresstype' in _.keys():
#                 if _['addresstype'] == 'city':
#                     return True

#     return False

# for city in ["new Dilli", "London", "Nandini", "Keshav"]:
#     print(f"Is {city} a city in India? {is_a_city(city)}.")

Okay so this did not work as expected. Is calling an API from Kaggle Notebooks not possible? I was trying to call OSM's API.

In [ ]:
category_counts = TRAINING_DATAFRAME['City'].value_counts()
print("Cities with a frequency of more than 50", category_counts[category_counts > 50].index.tolist())

In [ ]:
TRAINING_DATAFRAME["City"] = TRAINING_DATAFRAME["City"].apply(
    lambda x: x if x in ['Kalyan', 'Patna', 'Vasai-Virar', 'Kolkata', 'Ahmedabad', 'Meerut', 'Ludhiana', 'Pune', 'Rajkot', 'Visakhapatnam', 'Srinagar', 'Mumbai', 'Indore', 'Agra', 'Surat', 'Varanasi', 'Vadodara', 'Hyderabad', 'Kanpur', 'Jaipur', 'Thane', 'Lucknow', 'Nagpur', 'Bangalore', 'Chennai', 'Ghaziabad', 'Delhi', 'Bhopal', 'Faridabad', 'Nashik']
    else "UNKNOWN"
)

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["Sleep Duration"].value_counts().reset_index(), 
    x = "Sleep Duration", 
    y = "count", 
    labels={'index': 'Category', 'Category': 'Frequency'}, 
    title = "Frequency of Each Sleep Duration",
    template="plotly_dark").show()

Seriously, how can you sleep 40 hours!? 🤦 Let's consider only `Less than 5 hours`, `7-8 hours`, `More than 8 hours` and `5-6 hours`.

In [ ]:
TRAINING_DATAFRAME["Sleep Duration"] = TRAINING_DATAFRAME["Sleep Duration"].apply(
    lambda x: x if x in ["Less than 5 hours", "7-8 hours", "More than 8 hours", "5-6 hours"]
    else "UNKNOWN"
)

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["Dietary Habits"].value_counts().reset_index(), 
    x = "Dietary Habits", 
    y = "count", 
    labels={'index': 'Category', 'Category': 'Frequency'}, 
    title = "Frequency of Dietary Habits",
    template="plotly_dark").show()

Someone really messed up filling their data.

In [ ]:
TRAINING_DATAFRAME["Dietary Habits"] = TRAINING_DATAFRAME["Dietary Habits"].apply(
    lambda x: x if x in ["Moderate", "Unhealthy", "Healthy"]
    else "UNKNOWN"
)

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["Degree"].value_counts().reset_index(), 
    x = "Degree", 
    y = "count", 
    labels={'index': 'Category', 'Category': 'Frequency'}, 
    title = "Frequency of Degree",
    template="plotly_dark").show()

Lets Clean this data too

In [ ]:
category_counts = TRAINING_DATAFRAME['Degree'].value_counts()
print("Degrees with a frequency of more than 50", category_counts[category_counts > 50].index.tolist())

In [ ]:
TRAINING_DATAFRAME["Degree"] = TRAINING_DATAFRAME["Degree"].apply(
    lambda x: x if x in ['Class 12', 'B.Ed', 'B.Arch', 'B.Com', 'B.Pharm', 'BCA', 'M.Ed', 'MCA', 'BBA', 'BSc', 'MSc', 'LLM', 'M.Pharm', 'M.Tech', 'B.Tech', 'LLB', 'BHM', 'MBA', 'BA', 'ME', 'MD', 'MHM', 'BE', 'PhD', 'M.Com', 'MBBS', 'MA']
    else "UNKNOWN"
)

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["Profession"].value_counts().reset_index(), 
    x = "Profession", 
    y = "count", 
    labels={'index': 'Category', 'Category': 'Frequency'}, 
    title = "Frequency of Each Sleep Duration",
    template="plotly_dark").show()

In [ ]:
category_counts = TRAINING_DATAFRAME['Profession'].value_counts()
print("Profession with a frequency of more than 50", category_counts[category_counts > 50].index.tolist())

In [ ]:
TRAINING_DATAFRAME["Profession"] = TRAINING_DATAFRAME["Profession"].apply(
    lambda x: x if x in ['Student', 'Teacher', 'Working Professional', 'Content Writer', 'Architect', 'Consultant', 'HR Manager', 'Pharmacist', 'Doctor', 'Business Analyst', 'Entrepreneur', 'Chemist', 'Chef', 'Educational Consultant', 'Data Scientist', 'Researcher', 'Lawyer', 'Customer Support', 'Marketing Manager', 'Pilot', 'Travel Consultant', 'Plumber', 'Sales Executive', 'Manager', 'Judge', 'Electrician', 'Financial Analyst', 'Software Engineer', 'Civil Engineer', 'UX/UI Designer', 'Digital Marketer', 'Accountant', 'Finanancial Analyst', 'Mechanical Engineer', 'Graphic Designer', 'Research Analyst', 'Investment Banker']
    else "UNKNOWN"
)

In [ ]:
def __clean_dataframe(dataframe):
    dataframe["City"] = dataframe["City"].apply(
        lambda x: x if x in ['Kalyan', 'Patna', 'Vasai-Virar', 'Kolkata', 'Ahmedabad', 'Meerut', 'Ludhiana', 'Pune', 'Rajkot', 'Visakhapatnam', 'Srinagar', 'Mumbai', 'Indore', 'Agra', 'Surat', 'Varanasi', 'Vadodara', 'Hyderabad', 'Kanpur', 'Jaipur', 'Thane', 'Lucknow', 'Nagpur', 'Bangalore', 'Chennai', 'Ghaziabad', 'Delhi', 'Bhopal', 'Faridabad', 'Nashik']
        else "UNKNOWN"
    )
    dataframe["Sleep Duration"] = dataframe["Sleep Duration"].apply(
        lambda x: x if x in ["Less than 5 hours", "7-8 hours", "More than 8 hours", "5-6 hours"]
        else "UNKNOWN"
    )
    dataframe["Dietary Habits"] = dataframe["Dietary Habits"].apply(
        lambda x: x if x in ["Moderate", "Unhealthy", "Healthy"]
        else "UNKNOWN"
    )
    dataframe["Degree"] = dataframe["Degree"].apply(
        lambda x: x if x in ['Class 12', 'B.Ed', 'B.Arch', 'B.Com', 'B.Pharm', 'BCA', 'M.Ed', 'MCA', 'BBA', 'BSc', 'MSc', 'LLM', 'M.Pharm', 'M.Tech', 'B.Tech', 'LLB', 'BHM', 'MBA', 'BA', 'ME', 'MD', 'MHM', 'BE', 'PhD', 'M.Com', 'MBBS', 'MA']
        else "UNKNOWN"
    )
    dataframe["Profession"] = dataframe["Profession"].apply(
        lambda x: x if x in ['Student', 'Teacher', 'Working Professional', 'Content Writer', 'Architect', 'Consultant', 'HR Manager', 'Pharmacist', 'Doctor', 'Business Analyst', 'Entrepreneur', 'Chemist', 'Chef', 'Educational Consultant', 'Data Scientist', 'Researcher', 'Lawyer', 'Customer Support', 'Marketing Manager', 'Pilot', 'Travel Consultant', 'Plumber', 'Sales Executive', 'Manager', 'Judge', 'Electrician', 'Financial Analyst', 'Software Engineer', 'Civil Engineer', 'UX/UI Designer', 'Digital Marketer', 'Accountant', 'Finanancial Analyst', 'Mechanical Engineer', 'Graphic Designer', 'Research Analyst', 'Investment Banker']
        else "UNKNOWN"
    )
    
    return dataframe

TESTING_DATAFRAME = __clean_dataframe(TESTING_DATAFRAME)

In [ ]:
TRAINING_DATAFRAME.describe(include='object')

In [ ]:
categorical_columns = [
    "Gender",
    "City",
    "Working Professional or Student",
    "Profession",
    "Sleep Duration",
    "Dietary Habits",
    "Degree",
    "Have you ever had suicidal thoughts ?",
    "Family History of Mental Illness"
]


#encoded_columns = pandas.get_dummies(dataframe[categorical_column], prefix = categorical_column, prefix_sep = "__", dtype = int)
for categorical_column in categorical_columns:
#     for dataframe in TRAINING_DATAFRAME, TESTING_DATAFRAME:
    encoded_column = pandas.get_dummies(TRAINING_DATAFRAME[categorical_column], prefix = categorical_column, prefix_sep = "__", dtype = int)
    TRAINING_DATAFRAME = TRAINING_DATAFRAME.drop(categorical_column, axis = "columns")
    TRAINING_DATAFRAME = TRAINING_DATAFRAME.join(encoded_column)
    
    encoded_column = pandas.get_dummies(TESTING_DATAFRAME[categorical_column], prefix = categorical_column, prefix_sep = "__", dtype = int)
    TESTING_DATAFRAME = TESTING_DATAFRAME.drop(categorical_column, axis = "columns")
    TESTING_DATAFRAME = TESTING_DATAFRAME.join(encoded_column)

In [ ]:
training_id = TRAINING_DATAFRAME["id"]
training_name = TRAINING_DATAFRAME["Name"]
TRAINING_DATAFRAME = TRAINING_DATAFRAME.drop(["id", "Name"], axis = "columns")

testing_id = TESTING_DATAFRAME["id"]
testing_name = TESTING_DATAFRAME["Name"]
TESTING_DATAFRAME = TESTING_DATAFRAME.drop(["id", "Name"], axis = "columns")

Now it's time to Normalize the data.

In [ ]:
TRAINING_DATAFRAME.describe()

In [ ]:
normalizer = sklearn.preprocessing.StandardScaler()
normalizer_columns = [
            "Age", 
            "Academic Pressure", 
            "Work Pressure", 
            "CGPA", 
            "Study Satisfaction",
            "Job Satisfaction",
            "Work/Study Hours",
            "Financial Stress"
        ]

TRAINING_DATAFRAME[normalizer_columns] = pandas.DataFrame(
    normalizer.fit_transform(TRAINING_DATAFRAME[normalizer_columns])
)
TESTING_DATAFRAME[normalizer_columns] = pandas.DataFrame(
    normalizer.transform(TESTING_DATAFRAME[normalizer_columns])
)

## 3.2 🔍 Exploratory Data Analysis (EDA)

Let's make some quick graphs to see how the dataset looks like.

In [ ]:
data = {
    "Gender": ["Male", "Female", "Male", "Female"],
    "Count": [
        TRAINING_DATAFRAME[TRAINING_DATAFRAME["Gender__Male"] == 1].size,
        TRAINING_DATAFRAME[TRAINING_DATAFRAME["Gender__Female"] == 1].size,
        TESTING_DATAFRAME[TESTING_DATAFRAME["Gender__Male"] == 1].size,
        TESTING_DATAFRAME[TESTING_DATAFRAME["Gender__Female"] == 1].size
    ],
    "Dataset": ["Training", "Training", "Testing", "Testing"]
}

plotly.express.bar(
    data, 
    x = "Gender", 
    y = "Count", 
    color = "Dataset", 
    barmode = "group",
    title = "Distribution of Male and Female Entries in Training and Testing Datasets",
    template = "plotly_dark").show()

In [ ]:
# plotly.express.violin(
#     TRAINING_DATAFRAME,
#     x="Age",
#     template="plotly_dark",
#     title="Age Distribution as Rug Plot",
# )
fig = plotly.graph_objects.Figure()

fig.add_trace(plotly.graph_objects.Violin(
    x = TRAINING_DATAFRAME['Age'],
    name = "Training",
    box_visible = True,
    meanline_visible = True
))

fig.add_trace(plotly.graph_objects.Violin(
    x = TESTING_DATAFRAME['Age'],
    name = "Testing",
    box_visible = True,
    meanline_visible = True
))

fig.update_layout(
    template="plotly_dark",
    title="Age Distribution in Training and Testing Datasets",
    xaxis_title="Age",
)

fig.show()

In [ ]:
corr_matrix = TRAINING_DATAFRAME.corr()

fig = plotly.graph_objects.Figure(data = plotly.graph_objects.Heatmap(
    z = corr_matrix.values,
    x = corr_matrix.columns,
    y = corr_matrix.columns,
    colorscale = "Viridis",
    text = corr_matrix.values.round(2),
    texttemplate = "%{text}",
    hovertemplate = "Variable 1: %{y}<br>Variable 2: %{x}<br>Correlation: %{z:.2f}<extra></extra>"
))

fig.update_layout(
    title = "Correlation Matrix",
    xaxis = dict(title="Variables", showticklabels=False),
    yaxis = dict(title="Variables", showticklabels=False),
    height = 700,
    width = 700,
    template="plotly_dark"
)

fig.show()

MORE PLOTS COMING SOON

## 3.3 🤖 Model Building: Train machine learning models, starting with basic classifiers and then experimenting with more complex models as needed.

In [ ]:
def pipe(model, parameters, training_dataframe, testing_dataframe, testing_id, to_save):
    y = training_dataframe["Depression"]
    X = training_dataframe.drop("Depression", axis = "columns")
    
    classifier_ = sklearn.model_selection.GridSearchCV(
        model,
        parameters
    )
    
    classifier_.fit(X, y)
    
    best_params = classifier_.best_params_
    best_score = classifier_.best_score_
    
    print("Best Parameters:", best_params)
    print("Best Score:", best_score)
    
    if (to_save):
        out = pandas.DataFrame(classifier_.predict(testing_dataframe))
        res = pandas.concat([pandas.DataFrame(testing_id), out], axis = 1)
        res.columns = ["id", "Depression"]
        res[["id", "Depression"]].to_csv("submission.csv", index=False)

In [ ]:
models = [
    {
        "name": "RandomForestClassifier",
        "model": sklearn.ensemble.RandomForestClassifier(
            n_estimators = 100, 
            max_depth = 3, 
            random_state = 42
        ),
        "parameters": {
            'n_estimators': [50, 100, 150],
            'max_depth': [3, 5, 7],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        },
        "train": False,
        "save": False
    },
    {
        "name": "GradientBoostingClassifier",
        "model": sklearn.ensemble.GradientBoostingClassifier(loss='exponential', learning_rate = 0.1, n_estimators = 100),
        "parameters": {
            'loss': ('log_loss', ),
            'learning_rate': [0.1,],
            'n_estimators': [25, 50, 100, 200],
            'subsample': [1.0],
            'criterion': ('friedman_mse', ),
        },
        "train": False,
        "save": False
    },
    {
        "name": "CatBoostClassifier",
        "model": catboost.CatBoostClassifier(loss_function = 'Logloss', learning_rate = 0.1, iterations = 100, verbose = 0, random_seed = 42),
        "parameters": {
            'learning_rate': [0.01],
            'iterations': [2000],
            'depth': [12],
            'subsample': [1.0],
            'l2_leaf_reg': [10],
            'border_count': [128],
            'random_strength': [2],
            'boosting_type': ['Plain'],
            'bagging_temperature': [0.25]
        },
        "train": False,
        "save": True,
    },
    {
        "name": "LGBMClassifier",
        "model": lightgbm.LGBMClassifier(
            boosting_type = 'gbdt', 
            objective = 'binary', 
            learning_rate = 0.1, 
            n_estimators = 100, 
            random_state = 42,
            verbose = -1
        ),
        "parameters": {
            'learning_rate': [0.1],
            'n_estimators': [100, 1000],
            'max_depth': [7, 12],
            'subsample': [1.0],
            'num_leaves': [256]
        },
        "train": False,
        "save": True
    },
    {
        "name": "XGBClassifier",
        "model": xgboost.XGBClassifier(
            objective = 'binary:logistic', 
            learning_rate = 0.1, 
            n_estimators = 100, 
            max_depth = 3, 
            subsample = 1.0, 
            use_label_encoder = False, 
            random_state = 42
        ),
        "parameters": {
            'learning_rate': [0.1],
            'n_estimators': [1000, 1500, 2000],
            'max_depth': [7, 9, 11],
            'subsample': [1.0],
            'use_label_encoder': [False, True] 
        },
        "train": True,
        "save": True
    }
]

for model in models:
    if model["train"]:
        print(f"Running {model['name']} in pipeline")
        
        pipe(model["model"], model["parameters"], TRAINING_DATAFRAME, TESTING_DATAFRAME, testing_id, model["save"])